# CMN Sentences — Profanity Filter (token-based)

Filter `cmn_sentences.tsv`, save clean simplified output.

**Fixes vs old version** (old substring match wrongly deleted 老师/狗/电影/多 …, dropping ~30%):
1. **Purge the profane lists** — drop single-char terms (多/忙/干 …) and any term that is an HSK vocab word (= benign false positive).
2. **Token-based matching** — segment with jieba, drop a sentence only if a *whole token* is profane (no more substring carnage).

In [1]:
import csv
import re
from pathlib import Path

import jieba
import opencc

DATA_DIR = Path(".")
PROFANITY_DIR = DATA_DIR / "profanity-list"
HSK_DIR = DATA_DIR / "New HSK (2025)"
SENTENCES_FILE = DATA_DIR / "data/cmn_sentences.tsv"
OUTPUT_FILE = DATA_DIR / "data/cmn_sentences_clean.tsv"

HAN = re.compile(r"[\u4e00-\u9fff]")
TRAIL_DIGITS = re.compile(r"[0-9]+$")

t2s = opencc.OpenCC("t2s")   # traditional -> simplified

In [2]:
# HSK vocab (simplified) — used to spot benign false positives in the profane lists
HSK_FILES = [
    "HSK_Level_1_words.txt", "HSK_Level_2_words.txt", "HSK_Level_3_words.txt",
    "HSK_Level_4_words.txt", "HSK_Level_5_words.txt", "HSK_Level_6_words.txt",
    "HSK_Level_7-9_words.txt",
]
hsk_words = set()
for name in HSK_FILES:
    for line in open(HSK_DIR / name, encoding="utf-8"):
        w = t2s.convert(TRAIL_DIGITS.sub("", line.strip()))
        if w:
            hsk_words.add(w)
print(f"HSK vocab: {len(hsk_words)} words")

HSK vocab: 10896 words


In [3]:
def load_word_list(path: Path) -> set[str]:
    """One word per line, skip blanks/comments."""
    words = set()
    with open(path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            word = line.strip()
            if word and not word.startswith("#"):
                words.add(word)
    return words

def load_sexy_sensitive(path: Path) -> set[str]:
    """Format: `word,` per line — strip trailing comma."""
    words = set()
    with open(path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            word = line.strip().rstrip(",").strip()
            if word:
                words.add(word)
    return words

raw_profane: set[str] = set()
for p in PROFANITY_DIR.iterdir():
    if p.is_file() and p.name != "sexy-sensitive-words" and p.suffix != ".md":
        loaded = load_word_list(p)
        print(f"{p.name:40s}  {len(loaded):>5} words")
        raw_profane |= loaded

sexy_words = load_sexy_sensitive(PROFANITY_DIR / "sexy-sensitive-words")
print(f"{'sexy-sensitive-words':40s}  {len(sexy_words):>5} words")
raw_profane |= sexy_words
raw_profane = {t2s.convert(w) for w in raw_profane}
print(f"\nRaw unique profane terms (simplified): {len(raw_profane)}")

中文辱骂词-清理.txt                               2637 words
zh.txt                                      316 words
zh                                          294 words
SexHateLex.txt                             3016 words
words.txt                                  1811 words
zh1                                         318 words
sexy-sensitive-words                        304 words

Raw unique profane terms (simplified): 6548


In [4]:
# --- Purge the dirty lists ---
single_char = {w for w in raw_profane if len(w) == 1}
hsk_false_pos = {w for w in raw_profane if len(w) >= 2 and w in hsk_words}

profane_words = {w for w in raw_profane if len(w) >= 2 and w not in hsk_words}

print(f"dropped single-char terms : {len(single_char):>5}  e.g. {' '.join(sorted(single_char)[:20])}")
print(f"dropped HSK false-positives: {len(hsk_false_pos):>5}  e.g. {' '.join(sorted(hsk_false_pos)[:20])}")
print(f"profane terms kept         : {len(profane_words):>5}")

# Seed jieba so multi-char profane terms segment as single tokens
for w in profane_words:
    if HAN.search(w):
        jieba.add_word(w)

Building prefix dict from the default dictionary ...


Loading model from cache /tmp/claude-501/jieba.cache


dropped single-char terms :   128  e.g. 㞗 乳 交 产 低 信 倭 偷 傻 入 切 劫 卵 吊 呡 呷 呸 咬 哈 哼
dropped HSK false-positives:   402  e.g. 丑闻 专制 东北 中医 中国 中奖 乞丐 乞讨 乱七八糟 事件 井底之蛙 人权 人民币 仇恨 付出 传染 伤害 低下 住房 作弊
profane terms kept         :  6018


Loading model cost 0.222 seconds.


Prefix dict has been built successfully.


In [5]:
def is_profane(simplified_sentence: str, words: set[str]) -> bool:
    """Token-based: drop only if a whole segmented token is a profane term."""
    tokens = jieba.lcut(simplified_sentence.lower())
    return any(tok in words for tok in tokens)

lower_profane = {w.lower() for w in profane_words}

all_rows: list[tuple[str, str, str]] = []
with open(SENTENCES_FILE, encoding="utf-8") as f:
    for row in csv.reader(f, delimiter="\t"):
        if len(row) == 3:
            all_rows.append((row[0], row[1], t2s.convert(row[2])))

print(f"Total sentences loaded (simplified): {len(all_rows)}")

Total sentences loaded (simplified): 81790


In [6]:
clean_rows = [r for r in all_rows if not is_profane(r[2], lower_profane)]
removed = len(all_rows) - len(clean_rows)

print(f"Sentences removed : {removed:>6}")
print(f"Clean sentences   : {len(clean_rows):>6}")
print(f"Retention rate    : {len(clean_rows)/len(all_rows)*100:.1f}%")

# sanity: previously-nuked benign words must survive now
kept_text = {r[2] for r in clean_rows}
for w in ["老师", "狗", "电影", "弟弟"]:
    n = sum(1 for t in kept_text if w in t)
    print(f"  sentences containing {w}: {n}")

Sentences removed :   3521
Clean sentences   :  78269
Retention rate    : 95.7%
  sentences containing 老师: 408
  sentences containing 狗: 433
  sentences containing 电影: 283
  sentences containing 弟弟: 132


In [7]:
with open(OUTPUT_FILE, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerows(clean_rows)

print(f"Saved {len(clean_rows)} clean sentences \u2192 {OUTPUT_FILE}")

Saved 78269 clean sentences → data/cmn_sentences_clean.tsv
